# FluxPrint example: Paris (ICOS Cities) footprint climatology

This notebook computes footprints from a half-hourly eddy-covariance met table
with **FluxPrint** and compares them against the published **ICOS Cities**
footprint product for the same tower.

Requirements:

- internet access (the ICOS Cities data portal; the footprint product is a
  ~95 MB NetCDF, downloaded once below),
- the final map cell needs `folium` (`pip install fluxprint[viz]`).

In [ ]:
# 3rd-party modules
import pandas as pd
import numpy as np
from pathlib import Path
from matplotlib import pyplot as plt

# local modules
import fluxprint

## 1. Reference: the published ICOS Cities footprint product

Downloaded from the ICOS Cities data portal (the URL encodes the licence
acceptance). The file is ~95 MB; to skip the download on later runs, save it
next to this notebook as `paris_footprint_240424.nc`.

In [ ]:
local_copy = Path('paris_footprint_240424.nc')
if local_copy.exists():
    icos_cities_ffp = fluxprint.io.read_from_file(str(local_copy))
else:
    url = 'https://citydata.icos-cp.eu/licence_accept?ids=%5B%22F8xViMt9DnfYx7wFSfU-BsRS%22%5D'
    icos_cities_ffp = fluxprint.io.read_from_url(url)  # ~95 MB download
    # icos_cities_ffp.to_netcdf(local_copy)  # uncomment to cache locally
icos_cities_ffp

## 2. Input: the half-hourly met table

The Level-2 met data for the same station and period. FluxPrint matches
columns case-insensitively (`USTAR`, `WD`, `WS`, `TA`, `PA`, ...; `H`, the sensible heat
flux, is matched exactly to avoid FFP's lowercase `h`, the boundary-layer height); we
only build a proper `TIMESTAMP` column and align the period with the
footprint product.

In [ ]:
url = 'https://citydata.icos-cp.eu/licence_accept?ids=%5B%22bSV20lxhTszQobvLFm0KlCeU%22%5D'
icos_cities_data = fluxprint.io.read_from_url(url, na_values=['-9999'])
icos_cities_data['TIMESTAMP'] = pd.to_datetime(
    icos_cities_data.TIMESTAMP_START.astype(str)) - pd.Timedelta('1h')
icos_cities_data = icos_cities_data[np.isin(icos_cities_data.TIMESTAMP.dt.strftime('%y%m%d%H%M'),
                         icos_cities_ffp.timestep.astype(str))]
icos_cities_data.head()

## 3. Compute footprints with FluxPrint

One footprint per half hour (`by="TIMESTAMP"`, `aggregate=False`), on a 3 km
domain around the tower. `zm` is the **aerodynamic** measurement height
(z - d); site metadata supplies it here along with `z0` and a boundary-layer
height.

In [ ]:
series = fluxprint.wrapper(
    data=icos_cities_data, by="TIMESTAMP", aggregate=False,
    tower=(2.42222, 48.88514), tower_crs="EPSG:4326",   # (lon, lat)
    domain=[-1500, 1500, -1500, 1500], zm=95.5, z0=0.8, pblh=500,
)
# tower-centred LAEA, recorded in attrs
ds = series.georeference().to_xarray()
ds

## 4. Sanity check: footprint peak direction vs measured wind direction

The direction from the tower to each footprint's peak should track the
measured wind direction — for both the FluxPrint series (blue) and the ICOS
Cities product (red).

In [ ]:
def find_angle(point, middle):
    dx = point[0] - middle[0]
    dy = point[1] - middle[1]

    # Compute angle from x-axis (counter-clockwise), then convert:
    angle_rad = np.arctan2(dx, dy)
    angle_deg = np.degrees(angle_rad)

    # Convert to compass angle: 0 deg = North, increasing clockwise
    compass_angle = (90 - angle_deg) % 360
    return compass_angle

plt.figure(figsize=(12, 2))
plt.plot(icos_cities_data['TIMESTAMP'], icos_cities_data.WD, c='k')
plt.scatter(ds.time,
    [find_angle(fluxprint.utils.find_peak(f), (150, 150))
     for f in ds.footprint[:, :, :]], marker='o', c='blue')
plt.scatter(pd.to_datetime(icos_cities_ffp.timestep, format='%y%m%d%H%M'),
    [find_angle(fluxprint.utils.find_peak(f), (800, 800))
     for f in icos_cities_ffp.footprint[:, :, :]], marker='x', c='red')
plt.show()

## 5. Interactive map

Overlay both footprint climatologies on a street map (needs `folium`:
`pip install fluxprint[viz]`).

In [ ]:
fluxprint.utils.plot_leaflet(icos_cities_ffp, ds, labels=[
                             'ICOS Cities Product', 'FluxPrint'])